# Day 8: Performance Optimization

Today we'll learn how to optimize pandas operations for better performance and memory efficiency.

In [ ]:
import pandas as pd
import numpy as np
import time
import psutil
import os
from memory_profiler import profile

# Create large sample dataset
np.random.seed(42)
n_rows = 1000000

large_df = pd.DataFrame({
    'id': range(n_rows),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_rows),
    'value': np.random.randn(n_rows),
    'price': np.random.uniform(10, 1000, n_rows),
    'date': pd.date_range('2020-01-01', periods=n_rows, freq='1min'),
    'flag': np.random.choice([True, False], n_rows)
})

print(f"Dataset shape: {large_df.shape}")
print(f"Memory usage: {large_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 1. Memory Optimization

In [ ]:
# Check current memory usage
def memory_usage_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print("Original memory usage:")
print(large_df.dtypes)
print(f"Total memory: {memory_usage_mb(large_df):.2f} MB")
print()

# Optimize data types
optimized_df = large_df.copy()

# Convert category to categorical
optimized_df['category'] = optimized_df['category'].astype('category')

# Downcast numeric types
optimized_df['id'] = pd.to_numeric(optimized_df['id'], downcast='integer')
optimized_df['value'] = pd.to_numeric(optimized_df['value'], downcast='float')
optimized_df['price'] = pd.to_numeric(optimized_df['price'], downcast='float')

print("Optimized memory usage:")
print(optimized_df.dtypes)
print(f"Total memory: {memory_usage_mb(optimized_df):.2f} MB")
print(f"Memory reduction: {(1 - memory_usage_mb(optimized_df)/memory_usage_mb(large_df))*100:.1f}%")
print()

# Memory usage by column
memory_by_column = large_df.memory_usage(deep=True)
print("Memory usage by column (original):")
for col, mem in memory_by_column.items():
    print(f"{col}: {mem/1024**2:.2f} MB")
print()

memory_by_column_opt = optimized_df.memory_usage(deep=True)
print("Memory usage by column (optimized):")
for col, mem in memory_by_column_opt.items():
    print(f"{col}: {mem/1024**2:.2f} MB")

## 2. Efficient Data Loading

In [ ]:
# Save test data
large_df.to_csv('large_dataset.csv', index=False)

# Method 1: Standard loading
start_time = time.time()
df1 = pd.read_csv('large_dataset.csv')
time1 = time.time() - start_time
print(f"Standard loading: {time1:.2f} seconds")

# Method 2: Loading with optimized dtypes
dtypes = {
    'id': 'int32',
    'category': 'category',
    'value': 'float32',
    'price': 'float32',
    'flag': 'bool'
}

start_time = time.time()
df2 = pd.read_csv('large_dataset.csv', dtype=dtypes, parse_dates=['date'])
time2 = time.time() - start_time
print(f"Optimized loading: {time2:.2f} seconds")
print(f"Memory usage - Standard: {memory_usage_mb(df1):.2f} MB")
print(f"Memory usage - Optimized: {memory_usage_mb(df2):.2f} MB")
print()

# Method 3: Chunked loading
def load_in_chunks(filename, chunk_size=50000):
    chunks = []
    start_time = time.time()
    
    for chunk in pd.read_csv(filename, chunksize=chunk_size, dtype=dtypes, parse_dates=['date']):
        # Process chunk if needed
        chunks.append(chunk)
    
    result = pd.concat(chunks, ignore_index=True)
    load_time = time.time() - start_time
    return result, load_time

df3, time3 = load_in_chunks('large_dataset.csv')
print(f"Chunked loading: {time3:.2f} seconds")
print(f"Memory usage - Chunked: {memory_usage_mb(df3):.2f} MB")

# Method 4: Using specific columns only
start_time = time.time()
df4 = pd.read_csv('large_dataset.csv', usecols=['id', 'category', 'value'], dtype={'id': 'int32', 'category': 'category', 'value': 'float32'})
time4 = time.time() - start_time
print(f"Selective column loading: {time4:.2f} seconds")
print(f"Memory usage - Selective: {memory_usage_mb(df4):.2f} MB")

## 3. Vectorized Operations vs Loops

In [ ]:
# Create smaller dataset for timing comparisons
test_df = large_df.head(100000).copy()

# Method 1: Python loop (SLOW)
def calculate_with_loop(df):
    result = []
    for _, row in df.iterrows():
        if row['value'] > 0:
            result.append(row['price'] * 1.1)
        else:
            result.append(row['price'] * 0.9)
    return result

# Method 2: List comprehension (BETTER)
def calculate_with_list_comp(df):
    return [row['price'] * 1.1 if row['value'] > 0 else row['price'] * 0.9 
            for _, row in df.iterrows()]

# Method 3: Vectorized operation (BEST)
def calculate_vectorized(df):
    return np.where(df['value'] > 0, df['price'] * 1.1, df['price'] * 0.9)

# Method 4: Using pandas where
def calculate_pandas_where(df):
    return df['price'].where(df['value'] <= 0, df['price'] * 1.1) * 0.9

# Time comparisons (using smaller dataset)
small_df = test_df.head(10000)

print("Performance comparison (10,000 rows):")

# Vectorized (fastest)
start = time.time()
result_vec = calculate_vectorized(small_df)
time_vec = time.time() - start
print(f"Vectorized: {time_vec:.4f} seconds")

# List comprehension
start = time.time()
result_list = calculate_with_list_comp(small_df)
time_list = time.time() - start
print(f"List comprehension: {time_list:.4f} seconds ({time_list/time_vec:.1f}x slower)")

# Loop (slowest - only test with very small dataset)
tiny_df = small_df.head(1000)
start = time.time()
result_loop = calculate_with_loop(tiny_df)
time_loop = time.time() - start
print(f"Loop (1,000 rows): {time_loop:.4f} seconds (estimated {time_loop*10/time_vec:.1f}x slower for 10k rows)")

print()
print("Key takeaway: Use vectorized operations whenever possible!")

## 4. Efficient Groupby Operations

In [ ]:
# Compare different groupby approaches
test_df = optimized_df.head(500000)

# Method 1: Multiple separate groupby operations (SLOW)
def separate_groupby(df):
    mean_price = df.groupby('category')['price'].mean()
    sum_value = df.groupby('category')['value'].sum()
    count_records = df.groupby('category').size()
    return mean_price, sum_value, count_records

# Method 2: Single groupby with agg (FAST)
def combined_groupby(df):
    return df.groupby('category').agg({
        'price': 'mean',
        'value': 'sum',
        'id': 'count'
    })

# Method 3: Using transform for element-wise operations
def groupby_transform(df):
    df = df.copy()
    df['price_mean_by_category'] = df.groupby('category')['price'].transform('mean')
    df['price_deviation'] = df['price'] - df['price_mean_by_category']
    return df

print("Groupby performance comparison:")

# Separate groupby
start = time.time()
result1 = separate_groupby(test_df)
time1 = time.time() - start
print(f"Separate groupby: {time1:.4f} seconds")

# Combined groupby
start = time.time()
result2 = combined_groupby(test_df)
time2 = time.time() - start
print(f"Combined groupby: {time2:.4f} seconds ({time1/time2:.1f}x faster)")

# Transform operation
start = time.time()
result3 = groupby_transform(test_df)
time3 = time.time() - start
print(f"Groupby transform: {time3:.4f} seconds")

print()
print("Results preview:")
print(result2.head())

## 5. Index Optimization

In [ ]:
# Create test data
test_df = optimized_df.head(100000).copy()

# Without index
print("Performance with and without index:")

# Search without index
start = time.time()
result1 = test_df[test_df['category'] == 'A']
time1 = time.time() - start
print(f"Search without index: {time1:.4f} seconds")

# Set index and search
indexed_df = test_df.set_index('category')
start = time.time()
result2 = indexed_df.loc['A']
time2 = time.time() - start
print(f"Search with index: {time2:.4f} seconds ({time1/time2:.1f}x faster)")

# MultiIndex example
multi_indexed_df = test_df.set_index(['category', 'flag'])
start = time.time()
result3 = multi_indexed_df.loc[('A', True)]
time3 = time.time() - start
print(f"MultiIndex search: {time3:.4f} seconds")

# Sorted index benefits
date_df = test_df.copy()
date_df = date_df.sort_values('date').set_index('date')

start = time.time()
result4 = date_df.loc['2020-01-01':'2020-01-02']
time4 = time.time() - start
print(f"Date range search with sorted index: {time4:.4f} seconds")

print(f"\nFound {len(result4)} records in date range")

## 6. Efficient String Operations

In [ ]:
# Create string data
string_data = pd.DataFrame({
    'text': ['Hello World', 'Python Pandas', 'Data Science', 'Machine Learning'] * 25000,
    'category': ['A', 'B', 'C', 'D'] * 25000
})

print(f"String data shape: {string_data.shape}")
print(f"Memory usage: {memory_usage_mb(string_data):.2f} MB")

# Method 1: Regular string operations
start = time.time()
result1 = string_data['text'].str.upper()
time1 = time.time() - start
print(f"String upper(): {time1:.4f} seconds")

# Method 2: Using categorical for repeated strings
string_data_cat = string_data.copy()
string_data_cat['text'] = string_data_cat['text'].astype('category')
string_data_cat['category'] = string_data_cat['category'].astype('category')

print(f"Memory usage with categories: {memory_usage_mb(string_data_cat):.2f} MB")
print(f"Memory reduction: {(1 - memory_usage_mb(string_data_cat)/memory_usage_mb(string_data))*100:.1f}%")

# String operations on categorical
start = time.time()
result2 = string_data_cat['text'].str.upper()
time2 = time.time() - start
print(f"String upper() on categorical: {time2:.4f} seconds")

# Vectorized string operations
start = time.time()
contains_result = string_data['text'].str.contains('Python')
time3 = time.time() - start
print(f"String contains(): {time3:.4f} seconds")

# Extract patterns
start = time.time()
extract_result = string_data['text'].str.extract(r'(\w+)\s+(\w+)')
time4 = time.time() - start
print(f"String extract(): {time4:.4f} seconds")

print(f"\nExtracted patterns sample:")
print(extract_result.head())

## 7. Memory-Efficient Data Processing

In [ ]:
# Chunked processing for large datasets
def process_large_file_efficiently(filename, chunk_size=50000):
    """Process large file in chunks to save memory"""
    results = []
    total_processed = 0
    
    # Define dtypes for efficiency
    dtypes = {
        'id': 'int32',
        'category': 'category',
        'value': 'float32',
        'price': 'float32',
        'flag': 'bool'
    }
    
    for chunk in pd.read_csv(filename, chunksize=chunk_size, dtype=dtypes, parse_dates=['date']):
        # Process each chunk
        processed_chunk = chunk.groupby('category').agg({
            'price': ['mean', 'sum'],
            'value': 'count'
        })
        
        results.append(processed_chunk)
        total_processed += len(chunk)
        
        if total_processed % 100000 == 0:
            print(f"Processed {total_processed} rows...")
    
    # Combine results
    final_result = pd.concat(results).groupby(level=0).sum()
    return final_result

# Process the large file
print("Processing large file in chunks:")
start = time.time()
chunk_result = process_large_file_efficiently('large_dataset.csv')
chunk_time = time.time() - start
print(f"Chunked processing completed in {chunk_time:.2f} seconds")
print("\nResults:")
print(chunk_result)

# Memory-efficient operations
def memory_efficient_operations(df):
    """Demonstrate memory-efficient operations"""
    # Use inplace operations when possible
    df_copy = df.copy()
    
    # Instead of: df_copy = df_copy.dropna()
    df_copy.dropna(inplace=True)
    
    # Instead of: df_copy = df_copy.sort_values('price')
    df_copy.sort_values('price', inplace=True)
    
    # Use query for filtering (can be more memory efficient)
    filtered = df_copy.query('price > 500 and value > 0')
    
    return filtered

# Test memory-efficient operations
test_df = optimized_df.head(100000)
start = time.time()
efficient_result = memory_efficient_operations(test_df)
efficient_time = time.time() - start
print(f"\nMemory-efficient operations: {efficient_time:.4f} seconds")
print(f"Filtered dataset shape: {efficient_result.shape}")

## 8. Profiling and Benchmarking

In [ ]:
# Simple timing decorator
def time_it(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

# Benchmark different operations
@time_it
def operation_1(df):
    return df.groupby('category')['price'].mean()

@time_it
def operation_2(df):
    return df.pivot_table(values='price', index='category', aggfunc='mean')

@time_it
def operation_3(df):
    return df.groupby('category').agg({'price': 'mean'})['price']

test_df = optimized_df.head(200000)

print("Benchmarking different operations:")
result1 = operation_1(test_df)
result2 = operation_2(test_df)
result3 = operation_3(test_df)

# Memory profiling
def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # MB

print(f"\nCurrent memory usage: {get_memory_usage():.2f} MB")

# Profile a function
def memory_intensive_operation(df):
    # Create multiple copies (memory intensive)
    df1 = df.copy()
    df2 = df.copy()
    df3 = df.copy()
    
    # Perform operations
    result = pd.concat([df1, df2, df3])
    return result.groupby('category').sum()

print("\nMemory usage before operation:", f"{get_memory_usage():.2f} MB")
result = memory_intensive_operation(test_df.head(50000))
print("Memory usage after operation:", f"{get_memory_usage():.2f} MB")

# Clean up
del result
import gc
gc.collect()
print("Memory usage after cleanup:", f"{get_memory_usage():.2f} MB")

## 9. Parallel Processing

In [ ]:
# Using multiprocessing for pandas operations
from multiprocessing import Pool, cpu_count
import numpy as np

def process_chunk(chunk):
    """Process a chunk of data"""
    # Simulate some computation
    chunk['computed'] = chunk['price'] * chunk['value'] + np.sin(chunk['price'])
    return chunk.groupby('category')['computed'].sum()

def parallel_processing(df, n_processes=None):
    """Process dataframe in parallel"""
    if n_processes is None:
        n_processes = cpu_count()
    
    # Split dataframe into chunks
    chunk_size = len(df) // n_processes
    chunks = [df[i:i + chunk_size] for i in range(0, len(df), chunk_size)]
    
    # Process chunks in parallel
    with Pool(n_processes) as pool:
        results = pool.map(process_chunk, chunks)
    
    # Combine results
    final_result = pd.concat(results).groupby(level=0).sum()
    return final_result

def sequential_processing(df):
    """Process dataframe sequentially"""
    return process_chunk(df)

# Compare parallel vs sequential
test_df = optimized_df.head(200000)

print(f"Available CPU cores: {cpu_count()}")
print("\nComparing sequential vs parallel processing:")

# Sequential
start = time.time()
seq_result = sequential_processing(test_df)
seq_time = time.time() - start
print(f"Sequential processing: {seq_time:.4f} seconds")

# Parallel
start = time.time()
par_result = parallel_processing(test_df, n_processes=4)
par_time = time.time() - start
print(f"Parallel processing: {par_time:.4f} seconds")
print(f"Speedup: {seq_time/par_time:.2f}x")

print("\nResults comparison:")
print("Sequential:", seq_result.round(2))
print("Parallel:", par_result.round(2))
print("Results match:", np.allclose(seq_result.values, par_result.values))

## 10. Best Practices Summary

In [ ]:
# Performance optimization checklist
def optimize_dataframe(df):
    """Apply common optimizations to a dataframe"""
    optimized = df.copy()
    
    print("Original memory usage:", f"{memory_usage_mb(optimized):.2f} MB")
    
    # 1. Optimize data types
    for col in optimized.columns:
        col_type = optimized[col].dtype
        
        if col_type == 'object':
            # Check if it should be categorical
            unique_ratio = optimized[col].nunique() / len(optimized)
            if unique_ratio < 0.5:  # Less than 50% unique values
                optimized[col] = optimized[col].astype('category')
                print(f"Converted {col} to category (unique ratio: {unique_ratio:.3f})")
        
        elif col_type in ['int64', 'float64']:
            # Downcast numeric types
            if col_type == 'int64':
                optimized[col] = pd.to_numeric(optimized[col], downcast='integer')
            else:
                optimized[col] = pd.to_numeric(optimized[col], downcast='float')
            print(f"Downcasted {col} from {col_type} to {optimized[col].dtype}")
    
    print("Optimized memory usage:", f"{memory_usage_mb(optimized):.2f} MB")
    print(f"Memory reduction: {(1 - memory_usage_mb(optimized)/memory_usage_mb(df))*100:.1f}%")
    
    return optimized

# Apply optimizations
sample_df = large_df.head(100000)
optimized_sample = optimize_dataframe(sample_df)

print("\n" + "="*50)
print("PERFORMANCE OPTIMIZATION BEST PRACTICES")
print("="*50)

best_practices = [
    "1. Use appropriate data types (categorical, int32/float32 instead of int64/float64)",
    "2. Use vectorized operations instead of loops",
    "3. Combine multiple groupby operations into single agg() call",
    "4. Use indexes for frequent lookups",
    "5. Process large files in chunks",
    "6. Use inplace operations when possible",
    "7. Use query() for complex filtering",
    "8. Consider parallel processing for CPU-intensive tasks",
    "9. Profile your code to identify bottlenecks",
    "10. Clean up unused variables and use garbage collection"
]

for practice in best_practices:
    print(practice)

print("\n" + "="*50)
print("MEMORY OPTIMIZATION TIPS")
print("="*50)

memory_tips = [
    "• Use categorical for repeated string values",
    "• Downcast numeric types when possible",
    "• Process data in chunks for large datasets",
    "• Use sparse data structures for mostly-zero data",
    "• Delete intermediate variables when done",
    "• Use copy() only when necessary"
]

for tip in memory_tips:
    print(tip)

# Clean up large objects
del large_df, optimized_df
gc.collect()
print(f"\nFinal memory usage: {get_memory_usage():.2f} MB")

## Practice Exercises

In [ ]:
# Exercise 1: Optimize a poorly performing function
def slow_function(df):
    """Intentionally slow function for optimization practice"""
    results = []
    for category in df['category'].unique():
        subset = df[df['category'] == category]
        for _, row in subset.iterrows():
            if row['price'] > 500:
                results.append({
                    'category': category,
                    'high_price_count': 1,
                    'total_value': row['value']
                })
    return pd.DataFrame(results)

def fast_function(df):
    """Optimized version of the slow function"""
    high_price = df[df['price'] > 500]
    return high_price.groupby('category').agg({
        'price': 'count',
        'value': 'sum'
    }).rename(columns={'price': 'high_price_count', 'value': 'total_value'})

# Test with small dataset
test_data = pd.DataFrame({
    'category': np.random.choice(['A', 'B', 'C'], 10000),
    'price': np.random.uniform(100, 1000, 10000),
    'value': np.random.randn(10000)
})

print("Exercise 1: Function optimization")

# Time the slow function (with smaller dataset)
small_test = test_data.head(1000)
start = time.time()
slow_result = slow_function(small_test)
slow_time = time.time() - start
print(f"Slow function (1,000 rows): {slow_time:.4f} seconds")

# Time the fast function
start = time.time()
fast_result = fast_function(test_data)
fast_time = time.time() - start
print(f"Fast function (10,000 rows): {fast_time:.4f} seconds")
print(f"Estimated speedup: {(slow_time * 10) / fast_time:.1f}x")

# Exercise 2: Memory optimization challenge
def create_memory_efficient_dataset(n_rows=100000):
    """Create a memory-efficient dataset"""
    data = {
        'id': np.arange(n_rows, dtype='int32'),
        'category': pd.Categorical(np.random.choice(['A', 'B', 'C', 'D'], n_rows)),
        'subcategory': pd.Categorical(np.random.choice(['X', 'Y', 'Z'], n_rows)),
        'value': np.random.randn(n_rows).astype('float32'),
        'price': np.random.uniform(10, 1000, n_rows).astype('float32'),
        'is_active': np.random.choice([True, False], n_rows),
        'date': pd.date_range('2020-01-01', periods=n_rows, freq='1min')
    }
    return pd.DataFrame(data)

print("\nExercise 2: Memory-efficient dataset creation")
efficient_df = create_memory_efficient_dataset()
print(f"Memory usage: {memory_usage_mb(efficient_df):.2f} MB")
print("Data types:")
print(efficient_df.dtypes)

# Exercise 3: Chunked processing implementation
def process_in_chunks(df, chunk_size=10000, operation='mean'):
    """Process dataframe in chunks and combine results"""
    results = []
    
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        
        if operation == 'mean':
            chunk_result = chunk.groupby('category')['price'].mean()
        elif operation == 'sum':
            chunk_result = chunk.groupby('category')['price'].sum()
        elif operation == 'count':
            chunk_result = chunk.groupby('category').size()
        
        results.append(chunk_result)
    
    # Combine results appropriately
    if operation == 'mean':
        # For mean, we need to weight by counts
        combined = pd.concat(results, axis=1).mean(axis=1)
    else:
        combined = pd.concat(results, axis=1).sum(axis=1)
    
    return combined

print("\nExercise 3: Chunked processing")
chunked_result = process_in_chunks(efficient_df, chunk_size=20000, operation='mean')
direct_result = efficient_df.groupby('category')['price'].mean()

print("Chunked result:")
print(chunked_result.round(2))
print("Direct result:")
print(direct_result.round(2))
print(f"Results are close: {np.allclose(chunked_result.values, direct_result.values)}")

## Tomorrow's Preview
In Day 9, we'll cover:
- Advanced Pandas Techniques
- Custom Functions and Extensions
- Working with APIs and Web Data
- Integration with Other Libraries